# Entrega 3 · Despliegue para consumo
### Predicción de renuncias de empleados (IBM HR Analytics)

M71V/M72V 02 · Implementación de Modelos de Aprendizaje Automático · UBA · **Giovanni Raffo**

La primera celda solo actúa en Google Colab: descarga el proyecto e instala lo necesario.

In [1]:
import sys, os
if "google.colab" in sys.modules:
    if not os.path.exists("/content/attrition-uba"):
        !git clone -q https://github.com/gioaqp/attrition-uba.git /content/attrition-uba
    %cd /content/attrition-uba/notebooks
    !pip install -q scikit-learn==1.9.1 fastapi uvicorn

## 1. Qué se despliega

El modelo de la Entrega 2 quedó guardado en un archivo. Para que Recursos Humanos lo use no alcanza con el archivo: hace falta un **servicio** al que se le mandan los datos de un empleado y devuelve la respuesta.

Eso es `api/app.py`: un servicio web con dos direcciones.

| Dirección | Para qué sirve |
|---|---|
| `GET /health` | Preguntar si el servicio está vivo y si cargó el modelo. |
| `POST /predict` | Mandar los datos de un empleado y recibir `{"attrition": "Yes"/"No", "probabilidad": 0.99}`. |

**El punto central:** el servicio no vuelve a programar la preparación de los datos. Carga el mismo archivo `modelo_attrition.pkl` de la Entrega 2, que ya trae adentro la receta de preparación, el modelo y el punto de corte de 0,40. La transformación en producción es idéntica a la del entrenamiento porque **es el mismo objeto**.

En la computadora el servicio se levanta con `uvicorn api.app:app --reload` y se prueba en el navegador en `http://127.0.0.1:8000/docs`. Este cuaderno hace lo mismo sin navegador, para que se pueda ejecutar también en Colab.

## 2. Levantar el servicio

Se arranca el servicio en segundo plano y se le pregunta si está vivo.

In [2]:
import subprocess, sys, time, requests

URL = "http://127.0.0.1:8000"
servicio = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.app:app", "--port", "8000"], cwd="..")

for intento in range(30):
    try:
        estado = requests.get(f"{URL}/health").json()
        break
    except Exception:
        time.sleep(1)

print(estado)

{'estado': 'ok', 'modelo_cargado': True}


**Lectura:** el servicio responde y confirma que el modelo está cargado. Se carga **una sola vez** al arrancar, no en cada consulta.

## 3. Los tres empleados de prueba

`api/ejemplos.json` tiene tres empleados reales del grupo de examen, elegidos para mostrar los tres casos que Recursos Humanos va a ver: uno de riesgo alto, uno tranquilo y uno de frontera.

In [3]:
import json
import pandas as pd

ejemplos = json.load(open("../api/ejemplos.json", encoding="utf-8"))
pd.DataFrame(ejemplos).T[["Age", "JobRole", "OverTime", "BusinessTravel", "MonthlyIncome", "YearsAtCompany"]]

,Age,JobRole,OverTime,BusinessTravel,MonthlyIncome,YearsAtCompany
renuncia_probable,21,Sales Representative,Yes,Travel_Frequently,2174,3
estable,44,Manager,No,Travel_Rarely,19190,25
borderline,54,Sales Executive,No,Travel_Rarely,10686,9


## 4. Consultar el servicio

Se manda cada empleado con sus datos crudos, tal como salen de la tabla de Recursos Humanos, sin ningún paso manual de preparación.

In [4]:
for nombre, empleado in ejemplos.items():
    respuesta = requests.post(f"{URL}/predict", json=empleado).json()
    print(nombre, "->", respuesta)

renuncia_probable -> {'attrition': 'Yes', 'probabilidad': 0.9877}
estable -> {'attrition': 'No', 'probabilidad': 0.0358}
borderline -> {'attrition': 'Yes', 'probabilidad': 0.4374}


**Lectura:**

| Caso | Probabilidad | Respuesta | Qué pasó de verdad |
|---|---|---|---|
| Representante de ventas de 21 años, horas extra, viaja seguido, sueldo bajo | 0,99 | Yes | Renunció |
| Gerente de 44 años, sin horas extra, sueldo alto | 0,04 | No | Se quedó |
| Ejecutivo de ventas de 54 años, sin horas extra | 0,44 | Yes | Se quedó: falsa alarma |

El tercero es el caso interesante para la defensa: está apenas por encima del corte de 0,40. Con el corte en 0,50 no se marcaba. Esas falsas alarmas son el precio aceptado a cambio de detectar 8 de cada 10 renuncias reales: una conversación de más cuesta mucho menos que perder a un empleado sin haberlo visto venir.

## 5. Qué pasa si los datos vienen mal

El servicio valida la entrada antes de llegar al modelo: cada campo tiene su tipo y las siete variables de texto solo aceptan los valores que existen en el dataset.

In [5]:
malo = dict(ejemplos["estable"], OverTime="quizás")
respuesta = requests.post(f"{URL}/predict", json=malo)

print(respuesta.status_code)
print(respuesta.json()["detail"][0]["msg"])

422
Input should be 'No' or 'Yes'


**Lectura:** responde error 422 con el motivo exacto y el nombre del campo. El modelo nunca llega a ver un dato inválido.

## 6. Apagar el servicio

In [6]:
servicio.terminate()
print("Servicio detenido")

Servicio detenido


## 7. Cierre

| Punto de la consigna | Qué se hizo |
|---|---|
| API con `/predict` | FastAPI con `GET /health` y `POST /predict` |
| Formato de entrada y salida definido | 29 campos validados de entrada; salida `{"attrition", "probabilidad"}` |
| Mismo pipeline que en entrenamiento | Carga `modelo_attrition.pkl`, no reprograma nada |
| Probado con ejemplos | Tres empleados reales: riesgo alto, estable y de frontera |
| Documentado | Documentación interactiva automática en `/docs` |

Con esto quedan cubiertas las tres entregas: preparación de los datos, entrenamiento del modelo y puesta en funcionamiento.